# ADMM TEST

In [ ]:
import os
import time
import struct
import numpy as np
import scipy.sparse as sp

from pynq import Overlay, allocate, MMIO


# Scalar Parameters
alpha = 1.8
sigma = 1e-2

eps_abs = 5e-3
eps_rel = 5e-3

# Indirect solve accuracy control:
# - If MPC hits `pcg_max_iterations` almost every ADMM step, increase this.
pcg_tol_fraction = 1.0

admm_max_iterations = 2000
pcg_max_iterations = 5

# OSQP-style rho initialization (vectorized rho per constraint).
OSQP_RHO = 1.0
OSQP_RHO_MIN = 1e-6
OSQP_RHO_MAX = 1e6
OSQP_RHO_EQ_OVER_RHO_INEQ = 100
OSQP_RHO_TOL = 0.01

# =====================================================================
# 0. Hardware Dimensions (MUST MATCH YOUR HLS #defines)
# =====================================================================
PACK_SIZE = 16
TILE_ROWS = 8192  # Change this if your HLS MAX_ROWS is different
TILE_COLS = 8192  # Change this if your HLS MAX_COLS is different

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
bitstream_path = '/home/xilinx/admm/admm_fixed_tiles.bit' # Update if name changed

print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print("Overlay loaded!")

CONTROL_ADDR   = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE     = 0x10000

control_ip   = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

# =====================================================================
# 2. Load Canonicalized CVXPY QP Data & Scaling
# =====================================================================
data_dir = "./data"
print(f"\nLoading QP data from: {os.path.abspath(data_dir)}")

P_sparse = sp.load_npz(os.path.join(data_dir, "P.npz")).tocsc()
A_sparse = sp.load_npz(os.path.join(data_dir, "A.npz")).tocsc()
q    = np.load(os.path.join(data_dir, "q.npy")).astype(np.float32)
l_in = np.load(os.path.join(data_dir, "l.npy")).astype(np.float32)
u_in = np.load(os.path.join(data_dir, "u.npy")).astype(np.float32)

num_rows, num_cols = A_sparse.shape
print(f"Problem size: {num_rows} x {num_cols}")

P_diag_orig = P_sparse.diagonal().astype(np.float32)

print("\nApplying Scaling...")
def apply_scaling(P_diag, A_sparse, q, l, u, iters=10):
    n, m = len(P_diag), A_sparse.shape[0]
    E = np.ones(n, dtype=np.float32)
    D = np.ones(m, dtype=np.float32)
    P_scaled, A_scaled = P_diag.copy(), A_sparse.copy()

    for _ in range(iters):
        A_abs = abs(A_scaled)
        A_col_norm = A_abs.max(axis=0).toarray().flatten()
        x_norm = np.maximum(np.maximum(np.abs(P_scaled), A_col_norm), 1e-4)
        z_norm = np.maximum(A_abs.max(axis=1).toarray().flatten(), 1e-4)

        E_new, D_new = 1.0 / np.sqrt(x_norm), 1.0 / np.sqrt(z_norm)
        E *= E_new
        D *= D_new

        P_scaled = P_scaled * (E_new ** 2)
        A_scaled = sp.diags(D_new).dot(A_scaled).dot(sp.diags(E_new)).tocsc()

    q_scaled = q * E
    l_scaled = np.where(l == -np.inf, -np.inf, l * D)
    u_scaled = np.where(u ==  np.inf,  np.inf, u * D)

    c = 1.0 / np.maximum(np.max(np.abs(P_scaled)), np.max(np.abs(q_scaled)))
    c = max(c, 1e-4)
    return P_scaled * c, A_scaled, q_scaled * c, l_scaled, u_scaled, E, D, c

(P_diag_scaled, A_sparse_scaled, q_scaled, l_scaled, u_scaled, E_scale, D_scale, c_scale) = apply_scaling(P_diag_orig, A_sparse, q, l_in, u_in)

# Matrices for Tiling
AT_sparse = A_sparse_scaled.transpose().tocsc()
P_sparse_scaled = sp.diags(P_diag_scaled).tocsc() # Convert P to full CSC for SpMV

A_nnz = int(A_sparse_scaled.nnz)
P_nnz = int(P_sparse_scaled.nnz)

# =====================================================================
# 3. Python Helper Functions for Memory & Tiling
# =====================================================================
def ceil_div(a, b):
    return (a + b - 1) // b

def float_to_uint(f_val):
    return struct.unpack('<I', struct.pack('<f', float(f_val)))[0]

def uint_to_float(u_val):
    return struct.unpack('<f', struct.pack('<I', int(u_val)))[0]

def write_64bit_address(ip, base_offset, address):
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

def allocate_and_copy(np_array, dtype):
    """Allocates a PYNQ buffer and copies the numpy array data into it."""
    buf = allocate(shape=np_array.shape, dtype=dtype, cacheable=False)
    buf[:] = np_array[:]
    return buf

def build_tiled_csc(sp_mat, tile_rows, tile_cols, pack_size=16):
    """Slices a global SciPy CSC matrix into HW-ready packed tiles."""
    global_rows, global_cols = sp_mat.shape
    num_row_tiles = ceil_div(global_rows, tile_rows)
    num_col_tiles = ceil_div(global_cols, tile_cols)
    num_tiles = num_row_tiles * num_col_tiles

    g_col_ptr, g_row_idx, g_values = sp_mat.indptr, sp_mat.indices, sp_mat.data

    # Bucket non-zeros into tiles
    tiles_rows = [[[] for _ in range(tile_cols)] for _ in range(num_tiles)]
    tiles_vals = [[[] for _ in range(tile_cols)] for _ in range(num_tiles)]

    for c in range(global_cols):
        tc, local_c = c // tile_cols, c % tile_cols
        for idx in range(g_col_ptr[c], g_col_ptr[c+1]):
            r, v = g_row_idx[idx], g_values[idx]
            tr, local_r = r // tile_rows, r % tile_rows
            tile_idx = tr * num_col_tiles + tc
            tiles_rows[tile_idx][local_c].append(local_r)
            tiles_vals[tile_idx][local_c].append(v)

    # Flatten out into HW arrays
    tile_nnz_counts  = np.zeros(num_tiles, dtype=np.int32)
    tile_nnz_offsets = np.zeros(num_tiles, dtype=np.int32)
    tile_col_offsets = np.zeros(num_tiles, dtype=np.int32)
    col_ptr_tiled = []
    row_idx_tiled = []
    values_tiled  = []
    
    nnz_word_cursor = 0
    for tile_idx in range(num_tiles):
        tile_col_offsets[tile_idx] = len(col_ptr_tiled)
        
        local_col_ptr = [0] * (tile_cols + 1)
        tile_nnz = 0
        for c in range(tile_cols):
            tile_nnz += len(tiles_rows[tile_idx][c])
            local_col_ptr[c+1] = tile_nnz

        tile_nnz_counts[tile_idx] = tile_nnz
        tile_nnz_offsets[tile_idx] = nnz_word_cursor
        col_ptr_tiled.extend(local_col_ptr)

        # Pack values into 512-bit words (16 lanes)
        t_rows, t_vals = [], []
        for c in range(tile_cols):
            t_rows.extend(tiles_rows[tile_idx][c])
            t_vals.extend(tiles_vals[tile_idx][c])

        words = ceil_div(tile_nnz, pack_size)
        for w in range(words):
            row_word = np.zeros(pack_size, dtype=np.int32)
            val_word = np.zeros(pack_size, dtype=np.float32)
            for lane in range(pack_size):
                idx = w * pack_size + lane
                if idx < tile_nnz:
                    row_word[lane] = t_rows[idx]
                    val_word[lane] = t_vals[idx]
            row_idx_tiled.append(row_word)
            values_tiled.append(val_word)
        nnz_word_cursor += words

    # Return HW-ready numpy arrays
    return (num_row_tiles, num_col_tiles,
            np.array(tile_nnz_counts, dtype=np.int32),
            np.array(tile_nnz_offsets, dtype=np.int32),
            np.array(tile_col_offsets, dtype=np.int32),
            np.array(col_ptr_tiled, dtype=np.int32),
            np.array(row_idx_tiled, dtype=np.int32),
            np.array(values_tiled, dtype=np.float32))

# =====================================================================
# 4. Prepare and Allocate Hardware Buffers
# =====================================================================
print("\nSlicing and Allocating Tiled Matrices... (This may take a moment)")

buffers = [] # Keep track for cleanup

# --- Regular Matrix A (For Preconditioner) ---
def pack_csc_to_words(row_idx, values, num_words):
    nnz = len(row_idx)
    row_words = np.zeros((num_words, PACK_SIZE), dtype=np.int32)
    val_words = np.zeros((num_words, PACK_SIZE), dtype=np.float32)
    row_words.reshape(-1)[:nnz] = row_idx
    val_words.reshape(-1)[:nnz] = values
    return row_words, val_words

A_words = ceil_div(A_nnz, PACK_SIZE)
A_row_words_np, A_val_words_np = pack_csc_to_words(A_sparse_scaled.indices, A_sparse_scaled.data, A_words)

A_row_idx_reg_hw = allocate_and_copy(A_row_words_np, np.int32)
A_val_reg_hw     = allocate_and_copy(A_val_words_np, np.float32)
A_col_ptr_reg_hw = allocate_and_copy(A_sparse_scaled.indptr, np.int32)
buffers.extend([A_row_idx_reg_hw, A_val_reg_hw, A_col_ptr_reg_hw])

# --- Tiled Matrices ---
for name, sp_mat in [("A", A_sparse_scaled), ("AT", AT_sparse), ("P", P_sparse_scaled)]:
    print(f"  Tiling Matrix {name}...")
    (rtiles, ctiles, counts, noff, coff, cptr, ridx, vals) = build_tiled_csc(sp_mat, TILE_ROWS, TILE_COLS, PACK_SIZE)
    
    # Dynamically inject the metadata into the global namespace
    globals()[f"{name}_num_row_tiles"] = rtiles
    globals()[f"{name}_num_col_tiles"] = ctiles
    globals()[f"{name}_tile_nnz_counts_hw"]   = allocate_and_copy(counts, np.int32)
    globals()[f"{name}_tile_nnz_offsets_hw"]  = allocate_and_copy(noff, np.int32)
    globals()[f"{name}_tile_col_offsets_hw"]  = allocate_and_copy(coff, np.int32)
    globals()[f"{name}_col_ptr_tiled_hw"]     = allocate_and_copy(cptr, np.int32)
    
    if len(ridx) > 0:
        globals()[f"{name}_row_idx_tiled_hw"] = allocate_and_copy(ridx, np.int32)
        globals()[f"{name}_values_tiled_hw"]  = allocate_and_copy(vals, np.float32)
    else: # Fallback if matrix is completely empty
        globals()[f"{name}_row_idx_tiled_hw"] = allocate(shape=(1, PACK_SIZE), dtype=np.int32)
        globals()[f"{name}_values_tiled_hw"]  = allocate(shape=(1, PACK_SIZE), dtype=np.float32)

    buffers.extend([
        globals()[f"{name}_tile_nnz_counts_hw"], globals()[f"{name}_tile_nnz_offsets_hw"], globals()[f"{name}_tile_col_offsets_hw"],
        globals()[f"{name}_col_ptr_tiled_hw"], globals()[f"{name}_row_idx_tiled_hw"], globals()[f"{name}_values_tiled_hw"]
    ])

# --- Standard Dense Vectors ---
P_diag_hw = allocate_and_copy(P_diag_scaled, np.float32)
l_in_hw   = allocate_and_copy(l_scaled, np.float32)
u_in_hw   = allocate_and_copy(u_scaled, np.float32)
q_in_hw   = allocate_and_copy(q_scaled, np.float32)

rho_base = np.clip(OSQP_RHO, 1e-6, 1e6)
rho_in_np = np.full(num_rows, rho_base, dtype=np.float32)
rho_in_np[(np.isfinite(l_scaled)) & (np.isfinite(u_scaled)) & ((u_scaled - l_scaled) < 0.01)] = rho_base * 100
rho_in_np[(~np.isfinite(l_scaled)) & (~np.isfinite(u_scaled))] = 1e-6
rho_in_hw = allocate_and_copy(rho_in_np, np.float32)

x_out_hw = allocate(shape=(num_cols,), dtype=np.float32, cacheable=False)
y_out_hw = allocate(shape=(num_rows,), dtype=np.float32, cacheable=False)

buffers.extend([P_diag_hw, l_in_hw, u_in_hw, q_in_hw, rho_in_hw, x_out_hw, y_out_hw])

# =====================================================================
# 5. Hardware Execution
# =====================================================================
def run_hw(adaptive_rho):
    print(f"\n=== HW Run (adaptive_rho={adaptive_rho}) ===")

    

    # Write control scalars (Using the new Memory Map!)
    control_ip.write(0x10, num_rows)
    control_ip.write(0x18, num_cols)
    control_ip.write(0x20, A_nnz)
    control_ip.write(0x28, A_num_row_tiles)
    control_ip.write(0x30, A_num_col_tiles)
    control_ip.write(0x38, AT_num_row_tiles)
    control_ip.write(0x40, AT_num_col_tiles)
    control_ip.write(0x48, P_num_row_tiles)
    control_ip.write(0x50, P_num_col_tiles)

    control_ip.write(0x58, float_to_uint(sigma))
    control_ip.write(0x60, float_to_uint(alpha))
    control_ip.write(0x68, admm_max_iterations)
    control_ip.write(0x70, pcg_max_iterations)
    control_ip.write(0x78, int(adaptive_rho))
    control_ip.write(0x80, float_to_uint(eps_abs))
    control_ip.write(0x88, float_to_uint(eps_rel))
    control_ip.write(0x90, float_to_uint(pcg_tol_fraction))

    # Write Address Pointers (Using the new Memory Map!)
    # Regular A
    write_64bit_address(control_r_ip, 0x010, A_row_idx_reg_hw.device_address)
    print(f"A_row_idx physical addr: 0x{A_row_idx_reg_hw.device_address:016X}")
    write_64bit_address(control_r_ip, 0x01c, A_col_ptr_reg_hw.device_address)
    write_64bit_address(control_r_ip, 0x028, A_val_reg_hw.device_address)

    # Tiled A
    write_64bit_address(control_r_ip, 0x034, A_tile_nnz_counts_hw.device_address)
    write_64bit_address(control_r_ip, 0x040, A_tile_nnz_offsets_hw.device_address)
    write_64bit_address(control_r_ip, 0x04c, A_tile_col_offsets_hw.device_address)
    write_64bit_address(control_r_ip, 0x058, A_row_idx_tiled_hw.device_address)
    write_64bit_address(control_r_ip, 0x064, A_col_ptr_tiled_hw.device_address)
    write_64bit_address(control_r_ip, 0x070, A_values_tiled_hw.device_address)

    # Tiled AT
    write_64bit_address(control_r_ip, 0x07c, AT_tile_nnz_counts_hw.device_address)
    write_64bit_address(control_r_ip, 0x088, AT_tile_nnz_offsets_hw.device_address)
    write_64bit_address(control_r_ip, 0x094, AT_tile_col_offsets_hw.device_address)
    write_64bit_address(control_r_ip, 0x0a0, AT_row_idx_tiled_hw.device_address)
    write_64bit_address(control_r_ip, 0x0ac, AT_col_ptr_tiled_hw.device_address)
    write_64bit_address(control_r_ip, 0x0b8, AT_values_tiled_hw.device_address)

    # Tiled P
    write_64bit_address(control_r_ip, 0x0c4, P_tile_nnz_counts_hw.device_address)
    write_64bit_address(control_r_ip, 0x0d0, P_tile_nnz_offsets_hw.device_address)
    write_64bit_address(control_r_ip, 0x0dc, P_tile_col_offsets_hw.device_address)
    write_64bit_address(control_r_ip, 0x0e8, P_row_idx_tiled_hw.device_address)
    write_64bit_address(control_r_ip, 0x0f4, P_col_ptr_tiled_hw.device_address)
    write_64bit_address(control_r_ip, 0x100, P_values_tiled_hw.device_address)

    # Standard Vectors
    write_64bit_address(control_r_ip, 0x10c, P_diag_hw.device_address)
    write_64bit_address(control_r_ip, 0x118, l_in_hw.device_address)
    write_64bit_address(control_r_ip, 0x124, u_in_hw.device_address)
    write_64bit_address(control_r_ip, 0x130, q_in_hw.device_address)
    write_64bit_address(control_r_ip, 0x13c, rho_in_hw.device_address)
    write_64bit_address(control_r_ip, 0x148, x_out_hw.device_address)
    write_64bit_address(control_r_ip, 0x154, y_out_hw.device_address)

    # Start accelerator
    x_out_hw[:] = 0.0
    y_out_hw[:] = 0.0

    hw_start = time.time()
    control_ip.write(0x00, 0x01)
    while (control_ip.read(0x00) & 0x02) == 0:
        pass
    hw_end = time.time()

    # Read results (Using new map!)
    admm_iters = int(control_ip.read(0x98))
    pcg_iters  = int(control_ip.read(0xa8))
    r_prim_out = float(uint_to_float(control_ip.read(0xb8)))
    r_dual_out = float(uint_to_float(control_ip.read(0xc8)))
    status_out = int(control_ip.read(0xd8))

    avg_pcg = pcg_iters / admm_iters
    
    print(f"HW execution time: {(hw_end - hw_start) * 1000:.4f} ms")
    print(f"Status: {'Converged' if status_out == 1 else 'Max Iterations'}")
    print(f"ADMM Iterations: {admm_iters}")
    print(f"PCG Iterations : {pcg_iters} (Average pcg/admm: {avg_pcg})")
    print(f"Primal Residual: {r_prim_out:.5e}")
    print(f"Dual Residual  : {r_dual_out:.5e}")

    
    
    # Unscale and verify
    x_unscaled = np.array(x_out_hw) * E_scale
    obj = float(0.5 * np.sum(P_diag_orig * x_unscaled * x_unscaled) + np.dot(q, x_unscaled))
    viol = float(np.max(np.maximum(0.0, np.maximum(l_in - (A_sparse @ x_unscaled), (A_sparse @ x_unscaled) - u_in))))
    
    print(f"Objective Value: {obj:.6e}")
    print(f"Max Violation  : {viol:.3e}")
    
    return {
        "adaptive_rho": adaptive_rho,
        "status": status_out,
        "admm_iters": admm_iters,
        "pcg_iters": pcg_iters,
        "r_prim": r_prim_out,
        "r_dual": r_dual_out,
        "viol": viol,
        "obj": obj,
        "hw_ms": (hw_end - hw_start) * 1000.0,
    }

# Run both configs and collect results
RUN_BOTH_ADAPTIVE_RHO = True
adaptive_rho_list = [0, 1] if RUN_BOTH_ADAPTIVE_RHO else [0]
results = []

for adaptive in adaptive_rho_list:
    results.append(run_hw(adaptive))

# --- OUTSIDE THE LOOP (No indentation) ---
print("\n=== Summary ===")
if RUN_BOTH_ADAPTIVE_RHO and len(results) == 2:
    off = results[0]
    on = results[1]
    print(f"ADMM iterations: off={off['admm_iters']} | on={on['admm_iters']}")
    print(f"PCG iterations:  off={off['pcg_iters']} | on={on['pcg_iters']}")
    print(f"Primal residual: off={off['r_prim']:.3e} | on={on['r_prim']:.3e}")
    print(f"Dual residual:   off={off['r_dual']:.3e} | on={on['r_dual']:.3e}")
    print(f"Violation:       off={off['viol']:.3e} | on={on['viol']:.3e}")
    print(f"Objective:       off={off['obj']:.6e} | on={on['obj']:.6e}")
    print(f"HW time (ms):    off={off['hw_ms']:.3f} | on={on['hw_ms']:.3f}")
elif len(results) == 1:
    off = results[0]
    print(f"ADMM iterations: {off['admm_iters']}")
    print(f"PCG iterations:  {off['pcg_iters']}")
    print(f"Primal residual: {off['r_prim']:.3e}")
    print(f"Dual residual:   {off['r_dual']:.3e}")
    print(f"Violation:       {off['viol']:.3e}")
    print(f"Objective:       {off['obj']:.6e}")
    print(f"HW time (ms):    {off['hw_ms']:.3f}")


    
# Cleanup
for b in buffers:
    b.freebuffer()
print("\nBuffers released.")

Tuning:

alpha = 1.8
sigma = 1e-2

eps_abs = 1e-3
eps_rel = 1e-3

Indirect solve accuracy control:
If MPC hits `pcg_max_iterations` almost every ADMM step, increase this.
pcg_tol_fraction = 1.0

admm_max_iterations = 2000
pcg_max_iterations = 5

OSQP-style rho initialization (vectorized rho per constraint).
OSQP_RHO = 1.0
OSQP_RHO_MIN = 1e-6
OSQP_RHO_MAX = 1e6
OSQP_RHO_EQ_OVER_RHO_INEQ = 100
OSQP_RHO_TOL = 0.01